# Setup

In [1]:
import json
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import lightgbm as lgb
import time
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

c:\Users\madina\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, project_root)

from src.feature_eng import favorite_genre
from src.data_cleaning_parsing import extract_author_name

In [3]:
with open("../data/artifacts/als_full_model_info.pkl", "rb") as f:
    als_bundle = pickle.load(f)

In [4]:
X_als = als_bundle["user_factors"]      
Y_als = als_bundle["item_factors"]    
user_to_idx = als_bundle["user_to_idx"]
item_to_idx = als_bundle["item_to_idx"]
idx_to_user = als_bundle["idx_to_user"]
idx_to_item = als_bundle["idx_to_item"]
n_users = als_bundle["n_users"]
n_items = als_bundle["n_items"]

In [5]:
print(f"ALS loaded: X={X_als.shape}, Y={Y_als.shape}")

ALS loaded: X=(692133, 64), Y=(308122, 64)


In [6]:
with open("../data/artifacts/eval_setup.pkl", "rb") as f:
    setup = pickle.load(f)

In [7]:
val_user_idx = setup["user_ids"] 
gt_padded = setup["gt_padded"]
gt_sizes = setup["gt_sizes"]
K = setup["K"]
discounts = setup["discounts"]

In [8]:
print(f"val users: {len(val_user_idx)}, avg ground truth: {gt_sizes.mean():.2f}")

val users: 91497, avg ground truth: 1.78


In [9]:
df = pd.read_parquet("../data/interactions_final.parquet")
train = df[df["split"] == "train"].copy() 

In [10]:
meta = pd.read_parquet("../data/meta_books_filtered.parquet")
print(f"train: {len(train):,}, meta data: {len(meta):,}")

train: 3,230,426, meta data: 2,235,029


In [11]:
train["user_idx"] = train["user_id"].map(user_to_idx)
train["item_idx"] = train["parent_asin"].map(item_to_idx)
train["user_idx"] = train["user_idx"].astype(np.int32)
train["item_idx"] = train["item_idx"].astype(np.int32)

In [12]:
assert train["user_idx"].notna().all()
assert train["item_idx"].notna().all()

# Features

## User features

### Aggregation by user's interactions in train 

In [13]:
user_features = train.groupby("user_idx").agg(
    user_avg_rating=("rating", "mean"),
    user_n_interactions=("rating", "size"),
    user_rating_std=("rating", "std"),
    user_verified_mean=("verified_purchase", "mean"),
    user_helpful_mean=("helpful_vote_total", "mean"),
    user_helpful_max=("helpful_vote_total", "max"),
).reset_index()

In [14]:
user_features["user_rating_std"] = user_features["user_rating_std"].fillna(0)

In [15]:
print(f"user_features: {user_features.shape}")
user_features.head()

user_features: (692133, 7)


,user_idx,user_avg_rating,user_n_interactions,user_rating_std,user_verified_mean,user_helpful_mean,user_helpful_max
0,0,3.813253,166,1.006677,0.006024,1.76506,152.0
1,1,3.877551,147,0.508799,0.054422,0.77551,14.0
2,2,4.800000,10,0.632456,1.000000,0.30000,3.0
3,3,5.000000,3,0.000000,1.000000,0.00000,0.0
4,4,5.000000,5,0.000000,0.800000,0.60000,1.0


### User's favourite category

fav category is the one which a user has had most interactions with (left most reviews)

In [16]:
cat_df = pd.read_parquet("../data/artifacts/meta_categories.parquet")
print(f"categories for train items: {cat_df.shape}")
cat_df.head()

categories for train items: (308122, 5)


,parent_asin,genre_specific,genre_mid,categories_str,categories_list
0,0316185361,Leaders & Notable People,Biographies & Memoirs,"Books, Biographies & Memoirs, Leaders & Notabl...","[Books, Biographies & Memoirs, Leaders & Notab..."
1,B00KFOP3RG,Thrillers & Suspense,"Mystery, Thriller & Suspense","Books, Mystery, Thriller & Suspense, Thrillers...","[Books, Mystery, Thriller & Suspense, Thriller..."
2,1680450263,Engineering,Engineering & Transportation,"Books, Engineering & Transportation, Engineering","[Books, Engineering & Transportation, Engineer..."
3,1932225323,Schools & Teaching,Education & Teaching,"Books, Education & Teaching, Schools & Teaching","[Books, Education & Teaching, Schools & Teaching]"
4,B003P2WETK,Thrillers & Suspense,"Mystery, Thriller & Suspense","Books, Mystery, Thriller & Suspense, Thrillers...","[Books, Mystery, Thriller & Suspense, Thriller..."


In [17]:
train_genres = train[["user_idx", "parent_asin"]].merge(cat_df[["parent_asin", "genre_specific", "genre_mid"]], on="parent_asin", how="left")
train_genres["genre_specific"] = train_genres["genre_specific"].fillna("unknown")
train_genres["genre_mid"] = train_genres["genre_mid"].fillna("unknown")

print(f"train_genres: {train_genres.shape}")
print(f"missing specific genres: {(train_genres['genre_specific'] == 'unknown').mean():.2%}")

train_genres: (3230426, 4)
missing specific genres: 1.18%


In [18]:
train_genres.head(3)

,user_idx,parent_asin,genre_specific,genre_mid
0,0,1683690125,Genre Fiction,Literature & Fiction
1,0,1328948935,Science Fiction & Fantasy,Teen & Young Adult
2,1,0778369102,Thrillers & Suspense,"Mystery, Thriller & Suspense"


In [19]:
user_fav_spec = favorite_genre(train_genres, "user_idx", "genre_specific")
user_fav_mid = favorite_genre(train_genres, "user_idx", "genre_mid")

In [20]:
print(f"users with favorite specific genre: {len(user_fav_spec):,}")
print(user_fav_spec["fav_genre_specific"].value_counts().head(10))

users with favorite specific genre: 692,133
fav_genre_specific
Genre Fiction                 116285
Thrillers & Suspense           94534
Growing Up & Facts of Life     44426
Literature & Fiction           26282
Crafts & Hobbies               20672
Activities, Crafts & Games     20366
Americas                       18169
Action & Adventure             17337
Animals                        17028
Mystery                        15202
Name: count, dtype: int64


In [21]:
user_fav_spec[user_fav_spec['user_idx'] == 0]

,user_idx,fav_genre_specific
14,0,Literature & Fiction


In [22]:
user_fav_mid[user_fav_mid['user_idx'] == 0]

,user_idx,fav_genre_mid
12,0,Teen & Young Adult


frequency encoding

In [23]:
spec_freq = user_fav_spec["fav_genre_specific"].value_counts(normalize=True)
user_fav_spec["user_fav_spec_enc"] = (user_fav_spec["fav_genre_specific"].map(spec_freq).fillna(0))

In [24]:
user_fav_spec.head()

,user_idx,fav_genre_specific,user_fav_spec_enc
14,0,Literature & Fiction,0.037972
32,1,Genre Fiction,0.168010
41,2,Christian Living,0.013252
48,3,Crafts & Hobbies,0.029867
53,4,Thrillers & Suspense,0.136584


In [25]:
mid_freq = user_fav_mid["fav_genre_mid"].value_counts(normalize=True)
user_fav_mid["user_fav_mid_enc"] = (user_fav_mid["fav_genre_mid"].map(mid_freq).fillna(0))

merging to user_features

In [26]:
user_features = user_features.merge(user_fav_spec, on="user_idx", how="left")
user_features = user_features.merge(user_fav_mid, on="user_idx", how="left")

In [27]:
user_features.isna().sum()

user_idx               0
user_avg_rating        0
user_n_interactions    0
user_rating_std        0
user_verified_mean     0
user_helpful_mean      0
user_helpful_max       0
fav_genre_specific     0
user_fav_spec_enc      0
fav_genre_mid          0
user_fav_mid_enc       0
dtype: int64

In [28]:
print(f"user_features: {user_features.shape}")
user_features.head(3)

user_features: (692133, 11)


,user_idx,user_avg_rating,user_n_interactions,user_rating_std,user_verified_mean,user_helpful_mean,user_helpful_max,fav_genre_specific,user_fav_spec_enc,fav_genre_mid,user_fav_mid_enc
0,0,3.813253,166,1.006677,0.006024,1.76506,152.0,Literature & Fiction,0.037972,Teen & Young Adult,0.019037
1,1,3.877551,147,0.508799,0.054422,0.77551,14.0,Genre Fiction,0.168010,Literature & Fiction,0.193736
2,2,4.800000,10,0.632456,1.000000,0.30000,3.0,Christian Living,0.013252,Children's Books,0.215178


## Item features 

cleaning author column

In [29]:
item_meta = meta[["parent_asin", "title", "author", "price", "average_rating", "rating_number", "store"]].copy()

In [30]:
item_meta.isna().sum()

parent_asin            0
title                  0
author            785345
price             393564
average_rating         0
rating_number          0
store              73067
dtype: int64

In [31]:
item_meta['author_clean'] = item_meta['author'].apply(extract_author_name)

In [35]:
item_meta['author_clean'].value_counts().head(5)

author_clean
unknown                  785363
James Patterson            1422
DK                         1369
Charles River Editors       950
Stephen King                907
Name: count, dtype: int64

#